# Recommender System Baseline (ALS)
Базовый пайплайн для соревнования по рекомендательным системам. Используется модель ALS из библиотеки `implicit`.

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/andreypavlinich/bmstu-RecSys-2026/recommender-system-course-2026'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.parquet')
TEST_USERS_PATH = os.path.join(DATA_DIR, 'test_users.csv')
SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')


In [29]:
print('Загрузка train...')
train = pd.read_parquet(TRAIN_PATH)
test_users = pd.read_csv(TEST_USERS_PATH)
print(f'Train shape: {train.shape}')
print(f'Test users shape: {test_users.shape}')
print(train['event_type'].value_counts())


Загрузка train...
Train shape: (43000347, 7)
Test users shape: (9213, 1)
event_type
listen       41820481
like           781435
unlike         281924
dislike         97411
undislike       19096
Name: count, dtype: int64


In [ ]:
from implicit.nearest_neighbours import bm25_weight
from catboost import CatBoostRanker, Pool
from math import log2

def train_als_model(df_events, factors=128, iterations=15, half_life_ratio=0.25):
    """Строит взвешенную матрицу (события + time_decay + BM25) и обучает ALS."""
    event_weights = {
        'listen': 1.0,
        'like': 4.0,
        'unlike': -2.0,
        'dislike': -5.0,
        'undislike': 1.0
    }

    df = df_events.dropna(subset=['uid', 'item_id']).copy()
    df['uid'] = df['uid'].astype(int)
    df['item_id'] = df['item_id'].astype(int)

    if 'played_ratio_pct' in df.columns:
        ratio_mult = np.where(df['event_type'] == 'listen', df['played_ratio_pct'].fillna(50) / 50.0, 1.0)
    else:
        ratio_mult = 1.0

    max_ts = df['timestamp'].max()
    half_life = (df['timestamp'].max() - df['timestamp'].min()) * half_life_ratio
    time_decay = np.exp(-np.log(2) * (max_ts - df['timestamp']) / (half_life + 1e-5))

    df['weight'] = df['event_type'].map(event_weights).fillna(0.0) * ratio_mult * time_decay

    agg_df = df.groupby(['uid', 'item_id'], as_index=False)['weight'].sum()
    agg_df = agg_df[agg_df['weight'] > 0]

    user_ids = agg_df['uid'].unique()
    item_ids = agg_df['item_id'].unique()
    u2i = {uid: i for i, uid in enumerate(user_ids)}
    i2i = {iid: i for i, iid in enumerate(item_ids)}
    idx2i = {i: iid for iid, i in i2i.items()}

    rows = agg_df['uid'].map(u2i).values
    cols = agg_df['item_id'].map(i2i).values
    data = agg_df['weight'].values.astype(np.float32)

    matrix = sp.csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(item_ids)))
    matrix = bm25_weight(matrix, K1=100, B=0.8).tocsr() * 40

    model = AlternatingLeastSquares(
        factors=factors,
        regularization=0.01,
        iterations=iterations,
        random_state=42,
        use_gpu=False
    )
    model.fit(matrix, show_progress=True)

    return model, u2i, i2i, idx2i, matrix


def build_candidates_and_features(history_df, model, u2i_map, idx2i_map, matrix, target_uids=None, top_als=100, top_hist=50):
    """Собирает кандидатов из ALS и истории пользователя + считает признаки."""
    df = history_df.dropna(subset=['uid', 'item_id']).copy()
    df['uid'] = df['uid'].astype(int)
    df['item_id'] = df['item_id'].astype(int)

    if target_uids is not None:
        target_set = set(target_uids)
        df_users = df[df['uid'].isin(target_set)]
    else:
        df_users = df
        target_set = set(df['uid'].unique())

    max_t = df['timestamp'].max()

    user_item_stats = df_users.groupby(['uid', 'item_id']).agg(
        ui_count=('timestamp', 'count'),
        ui_last_ts=('timestamp', 'max'),
        ui_likes=('event_type', lambda x: (x == 'like').sum()),
        ui_dislikes=('event_type', lambda x: (x == 'dislike').sum())
    ).reset_index()
    user_item_stats['ui_recency'] = max_t - user_item_stats['ui_last_ts']

    item_pop = df.groupby('item_id').agg(
        item_total_cnt=('uid', 'count'),
        item_unique_users=('uid', 'nunique')
    ).reset_index()

    top_user_hist = (
        user_item_stats.sort_values(['uid', 'ui_count', 'ui_last_ts'], ascending=[True, False, False])
        .groupby('uid')
        .head(top_hist)[['uid', 'item_id']]
    )

    als_uids = [uid for uid in target_set if uid in u2i_map]
    als_records = []

    for uid in tqdm(als_uids, desc='ALS candidates'):
        u_idx = u2i_map[uid]
        ids, scores = model.recommend(u_idx, matrix[u_idx], N=top_als, filter_already_liked_items=False)
        for rank, (i_idx, score) in enumerate(zip(ids, scores)):
            als_records.append((uid, idx2i_map[i_idx], score, rank + 1))

    als_df = pd.DataFrame(als_records, columns=['uid', 'item_id', 'als_score', 'als_rank'])

    candidates = pd.concat([als_df[['uid', 'item_id']], top_user_hist], ignore_index=True).drop_duplicates()

    candidates = candidates.merge(als_df, on=['uid', 'item_id'], how='left')
    candidates['als_score'] = candidates['als_score'].fillna(-999.0)
    candidates['als_rank'] = candidates['als_rank'].fillna(999)

    candidates = candidates.merge(user_item_stats.drop(columns=['ui_last_ts']), on=['uid', 'item_id'], how='left')
    candidates[['ui_count', 'ui_likes', 'ui_dislikes']] = candidates[['ui_count', 'ui_likes', 'ui_dislikes']].fillna(0)
    candidates['ui_recency'] = candidates['ui_recency'].fillna(1e12)

    candidates = candidates.merge(item_pop, on='item_id', how='left').fillna(0)
    return candidates


def ndcg_at_k(recommended, actual, k=10):
    dcg = sum(1.0 / log2(i + 2) for i, item in enumerate(recommended[:k]) if item in actual)
    idcg = sum(1.0 / log2(i + 2) for i in range(min(len(actual), k)))
    return dcg / idcg if idcg > 0 else 0.0


In [ ]:
t1 = train['timestamp'].quantile(0.8)
t2 = train['timestamp'].quantile(0.9)

hist_80 = train[train['timestamp'] <= t1].copy()
target_10 = train[(train['timestamp'] > t1) & (train['timestamp'] <= t2)].copy()
hist_90 = train[train['timestamp'] <= t2].copy()
val_10 = train[train['timestamp'] > t2].copy()

print('--- Обучение ALS (80%) для тренировки бустинга ---')
als_80, u2i_80, _, idx2i_80, mat_80 = train_als_model(hist_80, factors=128, iterations=12)

train_cand = build_candidates_and_features(hist_80, als_80, u2i_80, idx2i_80, mat_80)

pos_events = target_10[target_10['event_type'].isin(['listen', 'like'])][['uid', 'item_id', 'event_type']].copy()
pos_events['target'] = np.where(pos_events['event_type'] == 'like', 2, 1)
pos_events = pos_events.groupby(['uid', 'item_id'], as_index=False)['target'].max()

train_cand = train_cand.merge(pos_events, on=['uid', 'item_id'], how='left')
train_cand['target'] = train_cand['target'].fillna(0).astype(int)

valid_train_uids = train_cand.groupby('uid')['target'].sum()
valid_train_uids = valid_train_uids[valid_train_uids > 0].index
train_cand = train_cand[train_cand['uid'].isin(valid_train_uids)].sort_values('uid')

feature_cols = [
    'als_score', 'als_rank', 'ui_count', 'ui_likes',
    'ui_dislikes', 'ui_recency', 'item_total_cnt', 'item_unique_users'
]

train_pool = Pool(
    data=train_cand[feature_cols],
    label=train_cand['target'],
    group_id=train_cand['uid'].values
)

print('--- Обучение CatBoostRanker ---')
ranker = CatBoostRanker(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='YetiRank',
    eval_metric='NDCG:top=10',
    random_seed=42,
    verbose=200
)
ranker.fit(train_pool)

print('--- Проверка качества на валидации (90% -> 10%) ---')
als_90, u2i_90, _, idx2i_90, mat_90 = train_als_model(hist_90, factors=128, iterations=12)

val_listen = val_10[val_10['event_type'].isin(['listen', 'like'])]
val_actuals = val_listen.groupby('uid')['item_id'].apply(set).to_dict()
val_uids = list(set(val_actuals.keys()) & set(hist_90['uid'].unique()))

val_cand = build_candidates_and_features(hist_90, als_90, u2i_90, idx2i_90, mat_90, target_uids=val_uids)
val_cand['pred'] = ranker.predict(val_cand[feature_cols])

val_top10 = (
    val_cand.sort_values(['uid', 'pred'], ascending=[True, False])
    .groupby('uid')['item_id']
    .apply(lambda x: list(x)[:10])
    .to_dict()
)

ndcg_scores = [ndcg_at_k(val_top10.get(uid, []), val_actuals[uid], k=10) for uid in val_uids]
print(f'Новый локальный NDCG@10 (ALS + CatBoost): {np.mean(ndcg_scores):.4f}')


--- Обучение ALS (80%) для тренировки бустинга ---


  0%|          | 0/12 [00:00<?, ?it/s]

ALS candidates: 100%|██████████| 9029/9029 [02:34<00:00, 58.41it/s]


--- Обучение CatBoostRanker ---
0:	total: 553ms	remaining: 9m 12s
200:	total: 2m 18s	remaining: 9m 11s
400:	total: 4m 47s	remaining: 7m 9s
600:	total: 7m 25s	remaining: 4m 55s
800:	total: 10m	remaining: 2m 29s
999:	total: 12m 54s	remaining: 0us
--- Проверка качества на валидации (90% -> 10%) ---


  0%|          | 0/12 [00:00<?, ?it/s]

ALS candidates: 100%|██████████| 8446/8446 [02:43<00:00, 51.52it/s]


Новый локальный NDCG@10 (ALS + CatBoost): 0.5063


In [ ]:
print('--- Финальное обучение ALS на 100% данных ---')
als_full, u2i_full, _, idx2i_full, mat_full = train_als_model(train, factors=128, iterations=15)

popular_items = (
    train[train['event_type'].isin(['listen', 'like'])]
    .groupby('item_id').size()
    .sort_values(ascending=False)
    .head(10).index.tolist()
)

test_user_ids = test_users['user_id'].tolist()
test_cand = build_candidates_and_features(train, als_full, u2i_full, idx2i_full, mat_full, target_uids=test_user_ids)
test_cand['pred'] = ranker.predict(test_cand[feature_cols])

test_top10 = (
    test_cand.sort_values(['uid', 'pred'], ascending=[True, False])
    .groupby('uid')['item_id']
    .apply(lambda x: list(x)[:10])
    .to_dict()
)

final_recs = []
for uid in test_user_ids:
    recs = test_top10.get(uid, [])
    if len(recs) < 10:
        for pop_item in popular_items:
            if pop_item not in recs:
                recs.append(pop_item)
            if len(recs) == 10:
                break
    final_recs.append(' '.join(map(str, recs[:10])))

submission = pd.DataFrame({
    'user_id': test_user_ids,
    'item_ids': final_recs
})

submission.to_csv('submission.csv', index=False)
print('Submission сохранен в submission.csv')
print(submission.head())


--- Финальное обучение ALS на 100% данных ---


  0%|          | 0/15 [00:00<?, ?it/s]

ALS candidates: 100%|██████████| 9060/9060 [02:48<00:00, 53.80it/s]


Submission сохранен в submission.csv
   user_id                                           item_ids
0        1  913405 535546 460034 181267 412529 894512 1028...
1        3  516512 576055 114312 153408 869426 282716 1566...
2        4  464601 433438 850071 353981 438073 356362 5390...
3        5  383685 668425 535546 33850 438073 159426 58794...
4        6  387671 13000 880532 162323 511035 416713 69806...


In [34]:
def validate_submission(sub_df, test_users_df):
    errors = []
    if len(sub_df) != len(test_users_df):
        errors.append(f'Несовпадение количества строк: {len(sub_df)} vs {len(test_users_df)}')
    if sub_df['item_ids'].isnull().any():
        errors.append('Найдены пустые значения в item_ids')
    for idx, row in sub_df.iterrows():
        items = str(row['item_ids']).strip().split()
        if len(items) == 0 or len(items) > 10:
            errors.append(f'User {row["user_id"]}: некорректное количество треков ({len(items)})')
        try:
            [int(i) for i in items]
        except ValueError:
            errors.append(f'User {row["user_id"]}: найдены нецелочисленные item_id')
    if errors:
        print('Найдены ошибки:')
        for e in errors[:10]:
            print(f'  - {e}')
    else:
        print('Валидация пройдена! Формат корректен.')

validate_submission(submission, test_users)


Валидация пройдена! Формат корректен.
